# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR<sup>2</sup>) using the `mlcroissant` library, referencing data elements by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` and `pandas` libraries are installed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# Display additional metadata (keywords, published date, cite as, etc.)
print(f"\nIdentifier: {metadata.identifier}")
print(f"Published on: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Keywords: {', '.join(metadata.keywords)}")
print(f"Cite as: {metadata.citeAs}")

## 2. Data Overview
Review available record sets and their field `@id`s. All references will use `@id` to maintain consistency and traceability.

In [ ]:
# List available record sets by @id - always reference by `@id`
from pprint import pprint

record_sets = dataset.record_sets
if len(record_sets) == 0:
    print("No record sets found directly in the 'record_sets' attribute.")
    # Fallback: try looking up in metadata if not attached
    if hasattr(metadata, 'record_sets') and metadata.record_sets:
        record_sets = metadata.record_sets
    else:
        # For the FAIR2 dataset, load and print the available record set ids
        # Try to explore through the dataset utility (using mlcroissant 0.4.x+ API style)
        from mlcroissant._dataset import _get_record_sets
        record_sets_meta = _get_record_sets(dataset._jsonld)
        record_sets = [rs['@id'] for rs in record_sets_meta]

print("Available record sets (`@id`):")
pprint(record_sets)

# Show fields for each record set (by @id if available) and the field @ids
fields_by_recordset = {}
for record_set_id in record_sets:
    print(f"\nFields in record set {record_set_id}:")
    fields = dataset.fields(record_set=record_set_id)
    field_ids = [f['@id'] for f in fields]
    print(field_ids)
    fields_by_recordset[record_set_id] = field_ids
# Store the first record set id for example usage: 
if record_sets:
    example_record_set_id = record_sets[0]
else:
    example_record_set_id = None

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract the data for the available record sets (by @id)
dataframes = {}
for record_set_id in record_sets:
    print(f"Extracting records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records with columns:")
        print(dataframes[record_set_id].columns.tolist())
    else:
        print("No records found.")

# Show preview of one of the record sets (first one):
df_key = example_record_set_id
if df_key and df_key in dataframes:
    print(f"\nHead of DataFrame for record set '{df_key}':")
    display(dataframes[df_key].head())
else:
    print("Could not load any DataFrame. Check record set IDs.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. These may include filtering records based on specific criteria, normalizing numeric fields, handling outliers, transforming distributions, or grouping data by relevant attributes, always referencing fields by their `@id`.

In [ ]:
# Select a record set and its numeric field for analysis (referenced by `@id`)
# For demonstration, use the first record set and search for a plausible numeric field
record_set_id = df_key
df = dataframes[record_set_id]

# Identify numeric fields by @id and dtype
numeric_field_id = None
for col in df.columns:
    # Heuristic: look for typical numeric fields, e.g. those containing 'age', 'interval', 'distance', 'count', or with dtype int/float
    if (df[col].dtype.kind in 'if') or any(x in col.lower() for x in ['age','interval','distance','count','number','score']):
        numeric_field_id = col
        break
if numeric_field_id is None:
    print("No numeric field found for EDA.")
else:
    print(f"Using numeric field '@id': {numeric_field_id}")
    # Handle possible missing/non-float conversion
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if not pd.isna(df[numeric_field_id].mean()) else 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    field_norm = f"{numeric_field_id}_normalized"
    filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, field_norm]].head())

    # Attempt to group by a categorical field (e.g. 'sex', 'category', 'site', 'status', etc.), using @id
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() < len(df) / 3 and df[col].dtype == object:
            group_field_id = col
            break
    if group_field_id:
        print(f"\nGrouped data by '{group_field_id}':")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df)
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if available, its relation to a selected grouping (categorical) field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    # Plot distribution of the numeric field
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field present, plot group comparison
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, explore, and perform basic analysis on the FAIR<sup>2</sup> tabular dataset using the `mlcroissant` library. 

Key steps included:
- Loading and summarizing metadata programmatically.
- Enumerating record sets and fields by their `@id`s.
- Extracting DataFrames for each record set, and referencing columns via `@id`.
- Performing simple EDA, including filtering, normalization, aggregation, and visualization.

This workflow can be used as a foundation for further, deeper domain-specific analysis.